# Data Assembly & Feature Engineering — Cáceres Solar Forecast

This notebook assembles the raw data into a single dataset suitable for TFT modelling.

**Steps:**
1. Load ERA5-Land weather data for each of the 8 PV plants
2. Compute solar position (zenith, azimuth) and clear-sky irradiance (GHI, DNI, DHI) via `pvlib`
3. Create cyclical calendar features (hour sin/cos, month sin/cos)
4. Capacity-weighted aggregation across plants → single regional weather signature
5. Merge with ESIOS regional PV generation target
6. Export the assembled dataset

In [2]:
import os
import glob
import numpy as np
import pandas as pd
import pvlib
from pvlib.location import Location

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

## 1. Plant metadata

The 8 largest PV plants in the province of Cáceres with their GPS coordinates and installed capacity.

In [3]:
PLANTS = {
    'Francisco_Pizarro': {'lat': 39.570, 'lon': -5.700, 'capacity_mwp': 589.88},
    'Tagus':             {'lat': 39.644, 'lon': -6.970, 'capacity_mwp': 379.79},
    'Cedillo':           {'lat': 39.627, 'lon': -7.461, 'capacity_mwp': 374.90},
    'Oriol':             {'lat': 39.8509,'lon': -6.747, 'capacity_mwp': 327.57},
    'Talasol':           {'lat': 39.628, 'lon': -6.310, 'capacity_mwp': 300.61},
    'Talayuela':         {'lat': 39.915, 'lon': -5.411, 'capacity_mwp': 300.00},
    'Arenales':          {'lat': 39.4533,'lon': -6.553, 'capacity_mwp': 150.31},
    'Campo_Aranuelo':    {'lat': 39.786, 'lon': -5.703, 'capacity_mwp':  39.95},
}

# Mapping from plant name -> ERA5 folder name (as they appear on disk)
PLANT_TO_FOLDER = {
    'Francisco_Pizarro': 'ERA5-LAND_Francisco_Pizarro_2022-2024',
    'Tagus':             'ERA5-LAND_Tagus_2022-2024',
    'Cedillo':           'ERA5-LAND_Cerdillo_2022-2024',  # note: folder is spelled "Cerdillo"
    'Oriol':             'ERA5-LAND_Oriol_2022-2024',
    'Talasol':           'ERA5-LAND_Talasol_2022-2024',
    'Talayuela':         'ERA5-LAND_Talayuela_2022-2024',
    'Arenales':          'ERA5-LAND_Arenales_2022-2024',
    'Campo_Aranuelo':    'ERA5-LAND_Campo_Aranuelo_2022-2024',
}

DATA_DIR = 'data'

total_capacity = sum(p['capacity_mwp'] for p in PLANTS.values())
print(f'Total installed capacity: {total_capacity:.2f} MWp across {len(PLANTS)} plants')
for name, info in PLANTS.items():
    pct = info['capacity_mwp'] / total_capacity * 100
    print(f'  {name:25s}  {info["capacity_mwp"]:7.2f} MWp  ({pct:5.1f}%)  @ ({info["lat"]:.3f}, {info["lon"]:.3f})')

Total installed capacity: 2463.01 MWp across 8 plants
  Francisco_Pizarro           589.88 MWp  ( 23.9%)  @ (39.570, -5.700)
  Tagus                       379.79 MWp  ( 15.4%)  @ (39.644, -6.970)
  Cedillo                     374.90 MWp  ( 15.2%)  @ (39.627, -7.461)
  Oriol                       327.57 MWp  ( 13.3%)  @ (39.851, -6.747)
  Talasol                     300.61 MWp  ( 12.2%)  @ (39.628, -6.310)
  Talayuela                   300.00 MWp  ( 12.2%)  @ (39.915, -5.411)
  Arenales                    150.31 MWp  (  6.1%)  @ (39.453, -6.553)
  Campo_Aranuelo               39.95 MWp  (  1.6%)  @ (39.786, -5.703)


## 2. Load ERA5-Land weather data

For each plant we load the three ERA5-Land CSV files (temperature, pressure/precipitation, radiation) and merge them on timestamp.

**Variables loaded and converted:**
- `t2m` -> 2m air temperature (K -> C)
- `d2m` -> 2m dewpoint temperature (K -> C)
- `sp` -> Surface pressure (Pa -> hPa)
- `tp` -> Total precipitation (m -> mm)
- `ssrd` -> Surface solar radiation downwards (J/m2 per hour -> W/m2)
- `strd` -> Surface thermal radiation downwards (J/m2 per hour -> W/m2)

In [4]:
def load_era5_plant(plant_name: str) -> pd.DataFrame:
    """Load and merge the 3 ERA5-Land CSVs for a given plant. Returns a DataFrame indexed by UTC datetime."""
    folder = os.path.join(DATA_DIR, PLANT_TO_FOLDER[plant_name])
    csvs = sorted(glob.glob(os.path.join(folder, '*.csv')))
    assert len(csvs) == 3, f'Expected 3 CSVs in {folder}, found {len(csvs)}'

    dfs = []
    for csv_path in csvs:
        df = pd.read_csv(csv_path, parse_dates=['valid_time'])
        df = df.set_index('valid_time').drop(columns=['latitude', 'longitude'])
        dfs.append(df)

    merged = pd.concat(dfs, axis=1)

    # Unit conversions
    merged['t2m'] = merged['t2m'] - 273.15          # K -> C
    merged['d2m'] = merged['d2m'] - 273.15          # K -> C
    merged['sp']  = merged['sp'] / 100.0            # Pa -> hPa
    merged['tp']  = merged['tp'] * 1000.0           # m -> mm
    merged['ssrd'] = merged['ssrd'] / 3600.0        # J/m2 (hourly) -> W/m2
    merged['strd'] = merged['strd'] / 3600.0        # J/m2 (hourly) -> W/m2

    # Rename for clarity
    merged = merged.rename(columns={
        't2m':  'temperature_2m_C',
        'd2m':  'dewpoint_2m_C',
        'sp':   'surface_pressure_hPa',
        'tp':   'total_precip_mm',
        'ssrd': 'ssrd_wm2',
        'strd': 'strd_wm2',
    })

    merged.index.name = 'datetime_utc'
    return merged


# Load all plants
era5_data = {}
for plant_name in PLANTS:
    era5_data[plant_name] = load_era5_plant(plant_name)
    print(f'{plant_name:25s}  {len(era5_data[plant_name]):6d} rows  '
          f'{era5_data[plant_name].index.min()} -> {era5_data[plant_name].index.max()}')

Francisco_Pizarro           26304 rows  2022-01-01 00:00:00 -> 2024-12-31 23:00:00
Tagus                       26304 rows  2022-01-01 00:00:00 -> 2024-12-31 23:00:00
Cedillo                     26304 rows  2022-01-01 00:00:00 -> 2024-12-31 23:00:00
Oriol                       26304 rows  2022-01-01 00:00:00 -> 2024-12-31 23:00:00
Talasol                     36552 rows  2022-01-01 00:00:00 -> 2026-03-03 23:00:00
Talayuela                   26304 rows  2022-01-01 00:00:00 -> 2024-12-31 23:00:00
Arenales                    26304 rows  2022-01-01 00:00:00 -> 2024-12-31 23:00:00
Campo_Aranuelo              26304 rows  2022-01-01 00:00:00 -> 2024-12-31 23:00:00


In [5]:
# Quick sanity check
era5_data['Francisco_Pizarro'].head(10)

,dewpoint_2m_C,temperature_2m_C,surface_pressure_hPa,total_precip_mm,ssrd_wm2,strd_wm2
datetime_utc,,,,,,
2022-01-01 00:00:00,4.77133,8.83563,968.08170,0.0,0.000000,272.924444
2022-01-01 01:00:00,4.31980,8.36624,967.71390,0.0,0.000000,272.252100
2022-01-01 02:00:00,3.74935,8.16693,967.66530,0.0,0.000000,271.687833
2022-01-01 03:00:00,3.17740,8.14733,967.60860,0.0,0.000000,270.989583
2022-01-01 04:00:00,2.58322,8.22915,967.44800,0.0,0.000000,270.219653
2022-01-01 05:00:00,2.11410,8.22463,966.94690,0.0,0.000000,269.517917
2022-01-01 06:00:00,1.85287,7.98434,966.93090,0.0,0.000000,268.448472
2022-01-01 07:00:00,1.84805,7.80682,967.86766,0.0,0.000000,269.922222
2022-01-01 08:00:00,1.65084,8.04520,967.84160,0.0,1.506111,269.168889


## 3. Compute solar position & clear-sky irradiance (pvlib)

For each plant, using its exact GPS coordinates, we compute:
- **Solar zenith angle** (degrees) — angle between the sun and vertical
- **Solar azimuth angle** (degrees) — compass direction of the sun
- **Clear-sky GHI, DNI, DHI** (W/m2) — theoretical irradiance under cloudless conditions using the Ineichen model with climatological Linke turbidity

These are computed at the same UTC timestamps as the ERA5 data.

In [6]:
def compute_solar_features(plant_name: str, timestamps: pd.DatetimeIndex) -> pd.DataFrame:
    """
    Compute solar position and clear-sky irradiance for a plant at given UTC timestamps.

    Returns DataFrame with columns:
        solar_zenith, solar_azimuth, clearsky_ghi, clearsky_dni, clearsky_dhi
    """
    info = PLANTS[plant_name]
    loc = Location(latitude=info['lat'], longitude=info['lon'], tz='UTC', altitude=400)
    # altitude ~400m is a reasonable average for the Caceres region

    # Ensure timestamps are UTC-localized
    if timestamps.tz is None:
        timestamps = timestamps.tz_localize('UTC')

    # Solar position
    solpos = loc.get_solarposition(timestamps)

    # Clear-sky irradiance (Ineichen model with climatological Linke turbidity)
    clearsky = loc.get_clearsky(timestamps, model='ineichen')

    result = pd.DataFrame({
        'solar_zenith':  solpos['apparent_zenith'].values,
        'solar_azimuth': solpos['azimuth'].values,
        'clearsky_ghi':  clearsky['ghi'].values,
        'clearsky_dni':  clearsky['dni'].values,
        'clearsky_dhi':  clearsky['dhi'].values,
    }, index=timestamps.tz_localize(None))  # store as tz-naive UTC

    result.index.name = 'datetime_utc'
    return result


# Compute for all plants
solar_data = {}
for plant_name in PLANTS:
    timestamps = era5_data[plant_name].index
    solar_data[plant_name] = compute_solar_features(plant_name, timestamps)
    cs_ghi_max = solar_data[plant_name]['clearsky_ghi'].max()
    print(f'{plant_name:25s}  max clear-sky GHI = {cs_ghi_max:.1f} W/m2')

solar_data['Francisco_Pizarro'].head(15)

Francisco_Pizarro          max clear-sky GHI = 961.0 W/m2
Tagus                      max clear-sky GHI = 965.4 W/m2
Cedillo                    max clear-sky GHI = 967.2 W/m2
Oriol                      max clear-sky GHI = 961.1 W/m2
Talasol                    max clear-sky GHI = 963.3 W/m2
Talayuela                  max clear-sky GHI = 954.9 W/m2
Arenales                   max clear-sky GHI = 967.2 W/m2
Campo_Aranuelo             max clear-sky GHI = 957.2 W/m2


,solar_zenith,solar_azimuth,clearsky_ghi,clearsky_dni,clearsky_dhi
datetime_utc,,,,,
2022-01-01 00:00:00,162.550200,339.587401,0.000000,0.000000,0.000000
2022-01-01 01:00:00,161.956553,25.953880,0.000000,0.000000,0.000000
2022-01-01 02:00:00,154.141635,57.168409,0.000000,0.000000,0.000000
2022-01-01 03:00:00,143.571985,74.588625,0.000000,0.000000,0.000000
2022-01-01 04:00:00,132.182212,86.296163,0.000000,0.000000,0.000000
2022-01-01 05:00:00,120.636942,95.726464,0.000000,0.000000,0.000000
2022-01-01 06:00:00,109.265238,104.367746,0.000000,0.000000,0.000000
2022-01-01 07:00:00,98.323294,113.037704,0.000000,0.000000,0.000000
2022-01-01 08:00:00,87.815598,122.318576,5.984406,82.563523,2.837433


## 4. Merge weather + solar features per plant

Combine the ERA5 weather data and pvlib solar features into a single DataFrame per plant.

In [7]:
plant_datasets = {}
for plant_name in PLANTS:
    combined = era5_data[plant_name].join(solar_data[plant_name], how='inner')
    plant_datasets[plant_name] = combined
    print(f'{plant_name:25s}  {len(combined)} rows, {combined.columns.tolist()}')

plant_datasets['Francisco_Pizarro'].head()

Francisco_Pizarro          26304 rows, ['dewpoint_2m_C', 'temperature_2m_C', 'surface_pressure_hPa', 'total_precip_mm', 'ssrd_wm2', 'strd_wm2', 'solar_zenith', 'solar_azimuth', 'clearsky_ghi', 'clearsky_dni', 'clearsky_dhi']
Tagus                      26304 rows, ['dewpoint_2m_C', 'temperature_2m_C', 'surface_pressure_hPa', 'total_precip_mm', 'ssrd_wm2', 'strd_wm2', 'solar_zenith', 'solar_azimuth', 'clearsky_ghi', 'clearsky_dni', 'clearsky_dhi']
Cedillo                    26304 rows, ['dewpoint_2m_C', 'temperature_2m_C', 'surface_pressure_hPa', 'total_precip_mm', 'ssrd_wm2', 'strd_wm2', 'solar_zenith', 'solar_azimuth', 'clearsky_ghi', 'clearsky_dni', 'clearsky_dhi']
Oriol                      26304 rows, ['dewpoint_2m_C', 'temperature_2m_C', 'surface_pressure_hPa', 'total_precip_mm', 'ssrd_wm2', 'strd_wm2', 'solar_zenith', 'solar_azimuth', 'clearsky_ghi', 'clearsky_dni', 'clearsky_dhi']
Talasol                    36552 rows, ['dewpoint_2m_C', 'temperature_2m_C', 'surface_pressure_hPa',

,dewpoint_2m_C,temperature_2m_C,surface_pressure_hPa,total_precip_mm,ssrd_wm2,strd_wm2,solar_zenith,solar_azimuth,clearsky_ghi,clearsky_dni,clearsky_dhi
datetime_utc,,,,,,,,,,,
2022-01-01 00:00:00,4.77133,8.83563,968.0817,0.0,0.0,272.924444,162.550200,339.587401,0.0,0.0,0.0
2022-01-01 01:00:00,4.31980,8.36624,967.7139,0.0,0.0,272.252100,161.956553,25.953880,0.0,0.0,0.0
2022-01-01 02:00:00,3.74935,8.16693,967.6653,0.0,0.0,271.687833,154.141635,57.168409,0.0,0.0,0.0
2022-01-01 03:00:00,3.17740,8.14733,967.6086,0.0,0.0,270.989583,143.571985,74.588625,0.0,0.0,0.0
2022-01-01 04:00:00,2.58322,8.22915,967.4480,0.0,0.0,270.219653,132.182212,86.296163,0.0,0.0,0.0


## 5. Capacity-weighted aggregation -> regional weather signature

Since our target variable is regional PV generation for all of Caceres, we aggregate the plant-level weather/solar features into a single set of regional features using **capacity-weighted averaging**.

Each plant's contribution is proportional to its installed capacity (MWp), so larger plants have more influence on the regional signature.

In [8]:
# Trim all plant datasets to the common time range (2022-01-01 to 2024-12-31)
common_start = '2022-01-01'
common_end   = '2024-12-31 23:00:00'

for plant_name in PLANTS:
    df = plant_datasets[plant_name]
    plant_datasets[plant_name] = df.loc[common_start:common_end]

# Verify all have the same index
ref_index = plant_datasets['Francisco_Pizarro'].index
for plant_name, df in plant_datasets.items():
    assert df.index.equals(ref_index), f'{plant_name} index mismatch!'
print(f'All plants aligned: {len(ref_index)} timestamps from {ref_index[0]} to {ref_index[-1]}')

All plants aligned: 26304 timestamps from 2022-01-01 00:00:00 to 2024-12-31 23:00:00


In [9]:
# Compute capacity weights (sum to 1)
total_cap = sum(p['capacity_mwp'] for p in PLANTS.values())
weights = {name: info['capacity_mwp'] / total_cap for name, info in PLANTS.items()}

print('Capacity weights:')
for name, w in weights.items():
    print(f'  {name:25s}  {w:.4f}  ({w*100:.1f}%)')
print(f'  Sum: {sum(weights.values()):.6f}')

Capacity weights:
  Francisco_Pizarro          0.2395  (23.9%)
  Tagus                      0.1542  (15.4%)
  Cedillo                    0.1522  (15.2%)
  Oriol                      0.1330  (13.3%)
  Talasol                    0.1220  (12.2%)
  Talayuela                  0.1218  (12.2%)
  Arenales                   0.0610  (6.1%)
  Campo_Aranuelo             0.0162  (1.6%)
  Sum: 1.000000


In [10]:
# Capacity-weighted average
feature_cols = plant_datasets['Francisco_Pizarro'].columns.tolist()

regional_weather = pd.DataFrame(0.0, index=ref_index, columns=feature_cols)
for plant_name, w in weights.items():
    regional_weather += plant_datasets[plant_name][feature_cols] * w

regional_weather.index.name = 'datetime_utc'
print(f'Regional weather signature: {regional_weather.shape}')
regional_weather.describe()

Regional weather signature: (26304, 11)


,dewpoint_2m_C,temperature_2m_C,surface_pressure_hPa,total_precip_mm,ssrd_wm2,strd_wm2,solar_zenith,solar_azimuth,clearsky_ghi,clearsky_dni,clearsky_dhi
count,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000
mean,8.623530,17.403121,973.073972,0.072826,200.540081,325.612514,89.682284,181.106488,229.626503,344.124497,34.323511
std,4.439559,8.268416,6.179866,0.281373,281.122281,40.709212,36.303488,100.681292,306.862402,384.807339,42.284021
min,-6.155613,-0.165251,944.797286,-0.000028,-0.000574,218.906512,17.222320,7.511998,0.000000,0.000000,0.000000
25%,5.613352,11.106330,969.263657,0.000000,0.000000,295.721080,61.044623,90.253506,0.000000,0.000000,0.000000
50%,9.006901,16.307123,972.376160,0.000000,10.763620,329.268363,89.742683,181.639272,0.276285,9.158572,0.169927
75%,12.030740,23.114995,976.768392,0.006602,362.370869,356.988440,118.340687,271.146333,453.585408,796.590745,68.828345
max,19.334168,42.215170,993.006751,5.952539,1007.698938,430.304658,163.051983,354.914704,962.228899,933.479659,126.931687


## 6. Cyclical calendar features

We encode temporal information as cyclical sine/cosine transforms so that e.g. December and January appear close together rather than 11 months apart.

- **Hour of day** -> `hour_sin`, `hour_cos` (period = 24)
- **Month of year** -> `month_sin`, `month_cos` (period = 12)

In [11]:
timestamps = regional_weather.index

# Hour of day (0-23)
hour = timestamps.hour
regional_weather['hour_sin'] = np.sin(2 * np.pi * hour / 24)
regional_weather['hour_cos'] = np.cos(2 * np.pi * hour / 24)

# Month of year (1-12)
month = timestamps.month
regional_weather['month_sin'] = np.sin(2 * np.pi * (month - 1) / 12)
regional_weather['month_cos'] = np.cos(2 * np.pi * (month - 1) / 12)

print('Calendar features added.')
regional_weather[['hour_sin', 'hour_cos', 'month_sin', 'month_cos']].describe()

Calendar features added.


,hour_sin,hour_cos,month_sin,month_cos
count,2.630400e+04,2.630400e+04,26304.000000,2.630400e+04
mean,-2.215044e-17,-5.672672e-17,-0.003014,-3.950846e-03
std,7.071202e-01,7.071202e-01,0.706952,7.072705e-01
min,-1.000000e+00,-1.000000e+00,-1.000000,-1.000000e+00
25%,-7.071068e-01,-7.071068e-01,-0.500000,-8.660254e-01
50%,6.123234e-17,-6.123234e-17,0.000000,-1.836970e-16
75%,7.071068e-01,7.071068e-01,0.866025,5.000000e-01
max,1.000000e+00,1.000000e+00,1.000000,1.000000e+00


## 7. Load and merge ESIOS target variable (regional PV generation)

The ESIOS data is Caceres regional solar generation downloaded from the ESIOS/REE platform.

The ESIOS timestamps are in CET/CEST (Europe/Madrid), while ERA5 is UTC. We convert ESIOS to UTC before merging. We use a **left join** on the weather data to preserve all timestamps — any hours missing from ESIOS will appear as NaN, which we can investigate during exploratory analysis.

In [12]:
esios_file = glob.glob(os.path.join(DATA_DIR, 'ESIOS*.csv'))
assert len(esios_file) == 1, f'Expected 1 ESIOS file, found {len(esios_file)}'

esios = pd.read_csv(esios_file[0], sep=';')
esios = esios[['datetime', 'value']].rename(columns={'value': 'pv_generation_gwh'})

# Parse the ISO datetime strings (which include timezone offsets like +01:00 / +02:00)
# and convert to UTC
esios['datetime_utc'] = pd.to_datetime(esios['datetime'], utc=True).dt.tz_localize(None)
esios = esios.set_index('datetime_utc').drop(columns=['datetime'])
esios = esios.sort_index()

print(f'ESIOS data: {len(esios)} rows from {esios.index[0]} to {esios.index[-1]}')
print(f'Generation range: {esios["pv_generation_gwh"].min():.3f} -- {esios["pv_generation_gwh"].max():.3f} GWh')
esios.head(10)

ESIOS data: 21502 rows from 2022-01-01 00:00:00 to 2024-12-31 22:00:00
Generation range: 0.001 -- 2821.934 GWh


,pv_generation_gwh
datetime_utc,
2022-01-01 00:00:00,0.004
2022-01-01 01:00:00,0.006
2022-01-01 02:00:00,0.010
2022-01-01 03:00:00,0.011
2022-01-01 04:00:00,0.007
2022-01-01 05:00:00,0.004
2022-01-01 07:00:00,3.258
2022-01-01 08:00:00,190.680
2022-01-01 09:00:00,621.527


In [13]:
# Merge weather features with generation target (left join to keep all weather timestamps)
dataset = regional_weather.join(esios, how='left')

n_total = len(dataset)
n_missing_target = dataset['pv_generation_gwh'].isna().sum()
n_present = n_total - n_missing_target

print(f'Final merged dataset: {n_total} rows x {dataset.shape[1]} columns')
print(f'Date range: {dataset.index[0]} to {dataset.index[-1]}')
print(f'\nESIOS target coverage: {n_present}/{n_total} hours present ({n_present/n_total*100:.1f}%)')
print(f'Missing target hours: {n_missing_target} ({n_missing_target/n_total*100:.1f}%) -- to investigate in EDA')
print(f'\nColumns ({len(dataset.columns)}):')
for col in dataset.columns:
    print(f'  {col}')

dataset.head()

Final merged dataset: 26304 rows x 16 columns
Date range: 2022-01-01 00:00:00 to 2024-12-31 23:00:00

ESIOS target coverage: 21502/26304 hours present (81.7%)
Missing target hours: 4802 (18.3%) -- to investigate in EDA

Columns (16):
  dewpoint_2m_C
  temperature_2m_C
  surface_pressure_hPa
  total_precip_mm
  ssrd_wm2
  strd_wm2
  solar_zenith
  solar_azimuth
  clearsky_ghi
  clearsky_dni
  clearsky_dhi
  hour_sin
  hour_cos
  month_sin
  month_cos
  pv_generation_gwh


,dewpoint_2m_C,temperature_2m_C,surface_pressure_hPa,total_precip_mm,ssrd_wm2,strd_wm2,solar_zenith,solar_azimuth,clearsky_ghi,clearsky_dni,clearsky_dhi,hour_sin,hour_cos,month_sin,month_cos,pv_generation_gwh
datetime_utc,,,,,,,,,,,,,,,,
2022-01-01 00:00:00,5.992456,9.045897,982.250519,0.0,0.0,278.746754,162.250513,337.709116,0.0,0.0,0.0,0.000000,1.000000,0.0,1.0,0.004
2022-01-01 01:00:00,5.598618,8.583755,981.807873,0.0,0.0,277.033911,162.080471,23.860362,0.0,0.0,0.0,0.258819,0.965926,0.0,1.0,0.006
2022-01-01 02:00:00,5.160802,8.362978,981.683232,0.0,0.0,275.748794,154.528562,55.919130,0.0,0.0,0.0,0.500000,0.866025,0.0,1.0,0.010
2022-01-01 03:00:00,4.679741,8.237647,981.617324,0.0,0.0,274.899749,144.057884,73.805948,0.0,0.0,0.0,0.707107,0.707107,0.0,1.0,0.011
2022-01-01 04:00:00,4.115510,8.022942,981.532890,0.0,0.0,273.865452,132.708566,85.723352,0.0,0.0,0.0,0.866025,0.500000,0.0,1.0,0.007


## 8. Quick data quality check

In [14]:
print('=== Missing values ===')
print(dataset.isnull().sum())
print(f'\nTotal missing: {dataset.isnull().sum().sum()}')

print('\n=== Data types ===')
print(dataset.dtypes)

print('\n=== Descriptive statistics ===')
dataset.describe()

=== Missing values ===
dewpoint_2m_C              0
temperature_2m_C           0
surface_pressure_hPa       0
total_precip_mm            0
ssrd_wm2                   0
strd_wm2                   0
solar_zenith               0
solar_azimuth              0
clearsky_ghi               0
clearsky_dni               0
clearsky_dhi               0
hour_sin                   0
hour_cos                   0
month_sin                  0
month_cos                  0
pv_generation_gwh       4802
dtype: int64

Total missing: 4802

=== Data types ===
dewpoint_2m_C           float64
temperature_2m_C        float64
surface_pressure_hPa    float64
total_precip_mm         float64
ssrd_wm2                float64
strd_wm2                float64
solar_zenith            float64
solar_azimuth           float64
clearsky_ghi            float64
clearsky_dni            float64
clearsky_dhi            float64
hour_sin                float64
hour_cos                float64
month_sin               float64
month_cos  

,dewpoint_2m_C,temperature_2m_C,surface_pressure_hPa,total_precip_mm,ssrd_wm2,strd_wm2,solar_zenith,solar_azimuth,clearsky_ghi,clearsky_dni,clearsky_dhi,hour_sin,hour_cos,month_sin,month_cos,pv_generation_gwh
count,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000,2.630400e+04,2.630400e+04,26304.000000,2.630400e+04,21502.000000
mean,8.623530,17.403121,973.073972,0.072826,200.540081,325.612514,89.682284,181.106488,229.626503,344.124497,34.323511,-2.215044e-17,-5.672672e-17,-0.003014,-3.950846e-03,617.450112
std,4.439559,8.268416,6.179866,0.281373,281.122281,40.709212,36.303488,100.681292,306.862402,384.807339,42.284021,7.071202e-01,7.071202e-01,0.706952,7.072705e-01,706.884800
min,-6.155613,-0.165251,944.797286,-0.000028,-0.000574,218.906512,17.222320,7.511998,0.000000,0.000000,0.000000,-1.000000e+00,-1.000000e+00,-1.000000,-1.000000e+00,0.001000
25%,5.613352,11.106330,969.263657,0.000000,0.000000,295.721080,61.044623,90.253506,0.000000,0.000000,0.000000,-7.071068e-01,-7.071068e-01,-0.500000,-8.660254e-01,0.027000
50%,9.006901,16.307123,972.376160,0.000000,10.763620,329.268363,89.742683,181.639272,0.276285,9.158572,0.169927,6.123234e-17,-6.123234e-17,0.000000,-1.836970e-16,278.409000
75%,12.030740,23.114995,976.768392,0.006602,362.370869,356.988440,118.340687,271.146333,453.585408,796.590745,68.828345,7.071068e-01,7.071068e-01,0.866025,5.000000e-01,1208.309750
max,19.334168,42.215170,993.006751,5.952539,1007.698938,430.304658,163.051983,354.914704,962.228899,933.479659,126.931687,1.000000e+00,1.000000e+00,1.000000,1.000000e+00,2821.934000


## 9. Export assembled dataset

In [15]:
output_path = os.path.join(DATA_DIR, 'caceres_assembled_dataset.csv')
dataset.to_csv(output_path)
print(f'Saved assembled dataset to: {output_path}')
print(f'Shape: {dataset.shape}')

Saved assembled dataset to: data/caceres_assembled_dataset.csv
Shape: (26304, 16)


## Summary

| Category | Features | Source |
|---|---|---|
| **Meteorological** | temperature_2m_C, dewpoint_2m_C, surface_pressure_hPa, total_precip_mm, ssrd_wm2, strd_wm2 | ERA5-Land (capacity-weighted avg) |
| **Solar position** | solar_zenith, solar_azimuth | pvlib (capacity-weighted avg) |
| **Clear-sky irradiance** | clearsky_ghi, clearsky_dni, clearsky_dhi | pvlib Ineichen model (capacity-weighted avg) |
| **Calendar (cyclical)** | hour_sin, hour_cos, month_sin, month_cos | Derived from timestamp |
| **Target** | pv_generation_gwh | ESIOS — Caceres regional solar generation |

**Total:** 15 independent variables + 1 target = 16 columns

